# Phase 1 and 2: Loading and cleaning the data

In this notebook we load all raw datasets from `data/raw/`, keep only the residential neighbourhoods with complete data, and save the results in `data/cleaned_data/` for the next notebooks.

In [1]:
import os                  # for folders and file paths
import pandas as pd        # for tables (DataFrames)
import geopandas as gpd    # for maps (DataFrame + geometry column)

# Folders for the raw input data and the cleaned output data
raw_folder = 'data/raw'
clean_folder = 'data/cleaned_data'

# Create the output folder making sure it does not create more floders if we run the notebook more often
if not os.path.exists(clean_folder):
    os.mkdir(clean_folder)

## Phase 1: Loading the raw data

### CBS tables
Both tables contain all regions (country, municipalities, districts, neighbourhoods). The region codes have spaces at the end, which would make linking the tables fail, so we remove them.

In [2]:
# Read both CBS tables from the raw data folder
kwb = pd.read_csv(os.path.join(raw_folder, 'kwb_2023.csv'))
nabijheid = pd.read_csv(os.path.join(raw_folder, 'nabijheid_2023.csv'))

kwb['Codering_3'] = kwb['Codering_3'].str.strip()               # .str.strip() removes the spaces around every value in the column
kwb['SoortRegio_2'] = kwb['SoortRegio_2'].str.strip()
nabijheid['Codering_3'] = nabijheid['Codering_3'].str.strip()

# Check the number of rows in both tables and how many regions of each type there are
print('kwb:', len(kwb), 'rows')
print('nabijheid:', len(nabijheid), 'rows')
kwb['SoortRegio_2'].value_counts()                              # number of regions per type

C:\Users\alecm\AppData\Local\Temp\ipykernel_47328\913031335.py:2: DtypeWarning: Columns (0: IndelingswijzigingGemeenteWijkBuurt_4) have mixed types. Specify dtype option on import or set low_memory=False.
  kwb = pd.read_csv(os.path.join(raw_folder, 'kwb_2023.csv'))


kwb: 18116 rows
nabijheid: 18116 rows


C:\Users\alecm\AppData\Local\Temp\ipykernel_47328\913031335.py:3: DtypeWarning: Columns (0: IndelingswijzigingWijkenEnBuurten_4) have mixed types. Specify dtype option on import or set low_memory=False.
  nabijheid = pd.read_csv(os.path.join(raw_folder, 'nabijheid_2023.csv'))


SoortRegio_2
Buurt       14421
Wijk         3352
Gemeente      342
Land            1
Name: count, dtype: int64

In [3]:
# Check that every region code in kwb also exists in nabijheid
codes_nabijheid = set(nabijheid['Codering_3'])      # a set is fast to search in

niet_gematcht = 0
for code in kwb['Codering_3']:
    if code not in codes_nabijheid:
        niet_gematcht += 1                          # count the codes that are not found

print('Regions in kwb without a match in nabijheid:', niet_gematcht)

Regions in kwb without a match in nabijheid: 0


### Stations and disruptions

In [4]:
# Read the stations file and print its length
stations = pd.read_csv(os.path.join(raw_folder, 'stations-2023-09-nl.csv'))
print('stations:', len(stations), 'rows')

# Read the disruption file of each year and collect them in a list
tabellen = []
for jaar in [2021, 2022, 2023]:
    file_path = os.path.join(raw_folder, 'disruptions-' + str(jaar) + '.csv')   # read the file of that year
    tabellen.append(pd.read_csv(file_path))

storingen = pd.concat(tabellen)                      # done with AI but putting it together in one table
print('storingen:', len(storingen), 'rows')

stations: 397 rows
storingen: 15541 rows


### Neighbourhood map

In [5]:
# Read only the 'buurten' layer of the CBS map, with three columns. 
buurten_kaart = gpd.read_file(
    os.path.join(raw_folder, 'wijkenbuurten_2023_v3.gpkg'),
    layer='buurten',
    columns=['buurtcode', 'buurtnaam', 'gemeentenaam'],
)
print('buurten_kaart:', len(buurten_kaart), 'rows')
print(buurten_kaart.crs)                          # EPSG:28992 = Dutch coordinate system in metres

c:\Users\alecm\anaconda3\envs\TIL6022-26\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


buurten_kaart: 14515 rows
EPSG:28992


## Phase 2: Neighbourhood filtering

### Select and rename the columns we need

In [ ]:
# Columns we keep from the CBS tables, and their new name, we rename them to shorter names for easier use in the code.
kwb_kolommen = {
    'Codering_3': 'buurtcode',
    'WijkenEnBuurten': 'buurtnaam',
    'Gemeentenaam_1': 'gemeente',
    'SoortRegio_2': 'soort_regio',
    'PersonenautoSPerHuishouden_112': 'autos_per_hh',
    'GemiddeldInkomenPerInwoner_81': 'inkomen',
    'MateVanStedelijkheid_125': 'stedelijkheid',
    'GemiddeldeHuishoudensgrootte_33': 'hh_grootte',
    'Bevolkingsdichtheid_34': 'dichtheid',
    'HuishoudensTotaal_29': 'n_huishoudens',
    'BedrijfsvestigingenTotaal_100': 'n_bedrijven',
}
nabijheid_kolommen = {
    'Codering_3': 'buurtcode',
    'AfstandTotTreinstationsTotaal_90': 'afstand_station',
    'AfstandTotBelangrijkOverstapstation_91': 'afstand_overstapstation',
}

# Select the columns we want to keep and rename them to shorter names for easier use in the code.
kwb_selectie = kwb[list(kwb_kolommen.keys())]                         # keep only the columns in the dictionary 
kwb_selectie = kwb_selectie.rename(kwb_kolommen, axis=1)              # give them the new names (axis=1 = columns)

#same for the nabijheid table
nabijheid_selectie = nabijheid[list(nabijheid_kolommen.keys())]
nabijheid_selectie = nabijheid_selectie.rename(nabijheid_kolommen, axis=1)

# Put both tables side by side: rows with the same buurtcode are matched
df = kwb_selectie.merge(nabijheid_selectie, on='buurtcode')
df['gemeente'] = df['gemeente'].str.strip()       # remove spaces around the names of the municipalities

df.head()

,buurtcode,buurtnaam,gemeente,soort_regio,autos_per_hh,inkomen,stedelijkheid,hh_grootte,dichtheid,n_huishoudens,n_bedrijven,afstand_station,afstand_overstapstation
0,NL00,Nederland,Nederland,Land,1.1,32.8,2.0,2.1,529.0,8270244,2358605,5.3,10.8
1,GM1680,Aa en Hunze,Aa en Hunze,Gemeente,1.4,32.8,5.0,2.2,93.0,11327,3180,12.4,14.4
2,WK168000,Wijk 00 Annen,Aa en Hunze,Wijk,1.3,34.1,5.0,2.3,246.0,1585,405,13.3,14.3
3,BU16800000,Annen,Aa en Hunze,Buurt,1.3,33.4,5.0,2.3,1369.0,1520,375,13.2,14.2
4,BU16800009,Verspreide huizen Annen,Aa en Hunze,Buurt,1.5,49.5,5.0,2.3,12.0,65,30,13.7,15.3


### Keep only neighbourhoods with complete data

In [ ]:
print('Start:', len(df))
# keep only neighbourhoods, not country/municipality/district
df = df[df['soort_regio'] == 'Buurt']              
print('Only neighbourhoods:', len(df))

df = df[df['autos_per_hh'].isna() == False]        # keep rows where car ownership is not missing
print('With car ownership:', len(df))

modelvariabelen = ['afstand_station', 'inkomen', 'stedelijkheid', 'hh_grootte', 'dichtheid']
for kolom in modelvariabelen:
    df = df[df[kolom].isna() == False]              # keep rows where this model variable is not missing
print('No missing model variables:', len(df))  

Start: 18116
Only neighbourhoods: 14421
With car ownership: 11447
No missing model variables: 11381


### Remove business areas
In business areas, company and lease cars are registered at only a few households, which makes car ownership look much higher. We find them with the number of businesses per household.

In [8]:
df['bedrijven_per_hh'] = df['n_bedrijven'] / df['n_huishoudens']   # new column, calculated row by row so that we can sort on it.

# Sort from low to high, so the neighbourhoods with the most businesses per household are at the bottom
df_gesorteerd = df.sort_values('bedrijven_per_hh')
df_gesorteerd[['buurtnaam', 'gemeente', 'n_huishoudens', 'bedrijven_per_hh', 'autos_per_hh']].tail(15)   # last 15 rows

,buurtnaam,gemeente,n_huishoudens,bedrijven_per_hh,autos_per_hh
15207,Havens en industrieterrein,Urk,175,2.200000,2.2
16207,Dobbewijk,Voorschoten,100,2.250000,1.9
17015,Bedrijventerrein Teylingen,Westland,120,2.250000,2.5
13839,Industrieterrein Soest,Soest,185,2.297297,2.5
1222,Geelvinckbuurt,Amsterdam,245,2.306122,0.4
17029,Bedrijventerrein Vlotlaan,Westland,95,2.315789,2.4
557,Gooisekant Oost,Almere,115,2.347826,1.8
6002,Tijlvoort-Noort,Goirle,50,2.400000,1.8
6388,Eemskanaal,Groningen,110,2.409091,1.9
7730,De Rietvelden oost,'s-Hertogenbosch,85,2.470588,1.6


In [ ]:
# we choice that a neighbourhood with more businesses than households is seen as a business area.
# This is about 3.5 times the national average, so normal residential neighbourhoods stay in.
DREMPEL = 1.0

aantal_voor = len(df)
df = df[df['bedrijven_per_hh'] <= DREMPEL]          # keep only neighbourhoods at or below the drempelvalue 

print('Removed as business area:', aantal_voor - len(df))
print('Neighbourhoods left:', len(df)) #check the number of neighbourhoods left after removing business areas

Removed as business area: 160
Neighbourhoods left: 11221


## Saving the clean data

In [10]:
df = df.drop('soort_regio', axis=1)                 # remove this column (axis=1): every row is a 'Buurt' now

# Save the tables as pickle files, so the next notebooks can load them quickly
df.to_pickle(os.path.join(clean_folder, 'buurten_schoon.pkl'))
stations.to_pickle(os.path.join(clean_folder, 'stations.pkl'))
storingen.to_pickle(os.path.join(clean_folder, 'storingen.pkl'))

# Save the map as a GeoPackage (.gpkg), so the shapes of the neighbourhoods are kept
buurten_kaart.to_file(os.path.join(clean_folder, 'buurten_kaart.gpkg'))